# Foody API Interactive Endpoint Playground

Use this notebook to test every endpoint in the current API interactively.

## Before you run
1. Start the API server (for example: uvicorn app.main:app --reload).
2. Make sure Google Places key is configured in your environment if you want restaurant discovery to return live data.
3. Run cells from top to bottom.

In [ ]:
import json
from typing import Any

import requests

BASE_URL = "http://127.0.0.1:8000"
TIMEOUT_SECONDS = 30

print(f"Using API base URL: {BASE_URL}")

Using API base URL: http://127.0.0.1:8001


In [3]:
def show_response(resp: requests.Response) -> dict[str, Any] | list[Any] | str:
    print(f"Status: {resp.status_code}")
    try:
        payload = resp.json()
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        return payload
    except ValueError:
        text = resp.text
        print(text)
        return text


def get(path: str):
    return requests.get(f"{BASE_URL}{path}", timeout=TIMEOUT_SECONDS)


def post(path: str, body: dict[str, Any]):
    return requests.post(f"{BASE_URL}{path}", json=body, timeout=TIMEOUT_SECONDS)

## 1) Health Check
Endpoint: GET /health

In [4]:
health = show_response(get("/health"))

Status: 200
{
  "status": "ok"
}


## 2) Nearby Restaurants
Endpoint: POST /restaurants/nearby

In [4]:
nearby_payload = {
    "location": {"lat": -37.8136, "lng": 144.9631},
    "radius": 2000
}

nearby_restaurants = show_response(post("/restaurants/nearby", nearby_payload))



ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8001): Max retries exceeded with url: /restaurants/nearby (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8001): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [13]:
first_restaurant_id = None
if isinstance(nearby_restaurants, list) and nearby_restaurants:
    first_restaurant_id = nearby_restaurants[0].get("id")

print(f"first_restaurant_id: {first_restaurant_id}")

first_restaurant_id: ChIJczgQh8lC1moR9r9gP44FRvY


## 3) Upsert User Profile
Endpoint: POST /users/profile

In [5]:
user_profile_payload = {
    "user_id": "demo_user_001",
    "goal_type": "muscle_gain",
    "cal_target": 2000,
    "macro_splits": {"protein": 0.3, "carbs": 0.4, "fat": 0.3},
    "restrictions": [],
    "budget_max": 30,
    "cuisine_preferences": ["Japanese", "Mediterranean"],
    "liked_items": [],
    "disliked_items": []
}

saved_profile = show_response(post("/users/profile", user_profile_payload))

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8001): Max retries exceeded with url: /users/profile (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8001): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

## 4) Get User Profile
Endpoint: GET /users/{user_id}

In [15]:
user_id = user_profile_payload["user_id"]
loaded_profile = show_response(get(f"/users/{user_id}"))

Status: 200
{
  "user_id": "demo_user_001",
  "goal_type": "muscle_gain",
  "cal_target": 2800.0,
  "macro_splits": {
    "protein": 0.35,
    "carbs": 0.45,
    "fat": 0.2
  },
  "restrictions": [],
  "budget_max": 30.0,
  "cuisine_preferences": [
    "Japanese",
    "Mediterranean"
  ],
  "liked_items": [],
  "disliked_items": []
}


## 5) Get Restaurant Menu
Endpoint: GET /restaurants/{restaurant_id}/menu

In [20]:
restaurant_menu = {}
selected_restaurant_id = None

if isinstance(nearby_restaurants, list) and nearby_restaurants:
    for restaurant in nearby_restaurants:
        rid = restaurant.get("id")
        if not rid:
            continue

        normalized_rid = rid.split("/")[-1]
        resp = get(f"/restaurants/{normalized_rid}/menu")
        print(f"Tried restaurant_id={normalized_rid} -> status={resp.status_code}")

        if resp.status_code != 200:
            continue

        payload = resp.json()
        items = payload.get("items", []) if isinstance(payload, dict) else []
        if items:
            selected_restaurant_id = normalized_rid
            restaurant_menu = payload
            break

    if selected_restaurant_id:
        print(f"Selected restaurant with menu items: {selected_restaurant_id}")
        print(json.dumps(restaurant_menu, indent=2, ensure_ascii=False))
    else:
        print("No non-empty menu found in first 5 nearby restaurants.")
        print("This is common when menu pages block scraping or providers have no menu URL.")
else:
    print("No nearby restaurants found. Check location/API key and rerun Cell 7.")

Tried restaurant_id=ChIJczgQh8lC1moR9r9gP44FRvY -> status=200
Tried restaurant_id=ChIJkZW8RMpC1moRixH-wBRZiRg -> status=200
Tried restaurant_id=ChIJg9OHO7RC1moR8LaOJz3PZok -> status=200
Selected restaurant with menu items: ChIJg9OHO7RC1moR8LaOJz3PZok
{
  "restaurant_id": "ChIJg9OHO7RC1moR8LaOJz3PZok",
  "items": [
    {
      "id": "2bba29d0-3122-4bbf-8d4d-f91b1777cf8a",
      "name": "Limited-Edition cannoli tasting box",
      "price": null,
      "description": "Special cannoli flavours for Cannoli Week.",
      "category": "Pastry Box",
      "tags": [],
      "estimated_calories_kcal": 250,
      "estimated_protein_g": 5.0,
      "nutrition_confidence": "medium",
      "nutrition_notes": "Estimate for one cannoli from the tasting box."
    },
    {
      "id": "d12cffff-a166-4dc2-a35c-995a5f8d7198",
      "name": "Brunetti Oro Rocher",
      "price": 57.0,
      "description": null,
      "category": "Cake",
      "tags": [],
      "estimated_calories_kcal": 450,
      "estimated_

## 6) Batch Nutrition Estimation
Endpoint: POST /menu/nutrition

In [9]:
menu_items = []
if isinstance(restaurant_menu, dict):
    menu_items = restaurant_menu.get("items", [])

if not menu_items:
    menu_items = [
        {
            "id": "demo_item_1",
            "name": "Chicken Salad",
            "price": 12.5,
            "description": "Grilled chicken, mixed greens, olive oil dressing",
            "category": "Salads",
            "tags": ["high-protein"]
        }
    ]

nutrition_payload = {
    "items": menu_items[:3],
    "estimator": "ai"
}

nutrition_result = show_response(post("/menu/nutrition", nutrition_payload))

Status: 200
{
  "items": [
    {
      "id": "demo_item_1",
      "calories": null,
      "protein": null,
      "carbs": null,
      "fat": null,
      "confidence": "estimated"
    }
  ]
}


## 7) Recommendations
Endpoint: POST /recommendations

In [21]:
recommendation_result = {}
target_restaurant_id = selected_restaurant_id or first_restaurant_id

if target_restaurant_id:
    recommendation_payload = {
        "user_id": user_id,
        "restaurant_id": target_restaurant_id,
        "top_n": 5
    }
    recommendation_result = show_response(post("/recommendations", recommendation_payload))
else:
    print("No restaurant ID available for recommendation test.")

Status: 200
{
  "restaurant_id": "ChIJg9OHO7RC1moR8LaOJz3PZok",
  "recommendations": []
}


## 8) Discover (Queued + Polling)
Endpoints: POST /discover, GET /discover/jobs/{job_id}

In [1]:
import time

discover_payload = {
    "location": nearby_payload["location"],
    "radius": 2000,  # nearby_payload["radius"]
    "profile": user_profile_payload,
    "top_n": 6
}

enqueue_result = show_response(post("/discover", discover_payload))

discover_job_id = None
if isinstance(enqueue_result, dict):
    discover_job_id = enqueue_result.get("job_id")

discover_result = {}
if discover_job_id:
    print(f"Polling discover job: {discover_job_id}")
    for attempt in range(1, 31):
        job_resp = get(f"/discover/jobs/{discover_job_id}")
        if job_resp.status_code != 200:
            print(f"Poll failed (status={job_resp.status_code})")
            show_response(job_resp)
            break

        job_payload = job_resp.json()
        status = job_payload.get("status")
        print(f"Attempt {attempt}: status={status}")

        if status == "completed":
            discover_result = job_payload.get("result", {})
            break
        if status == "failed":
            print(f"Discover job failed: {job_payload.get('error')}")
            break

        time.sleep(1.0)
else:
    print("No discover job id returned.")

discover_recommendations = []
if isinstance(discover_result, dict):
    discover_recommendations = discover_result.get("recommendations", [])

print(f"Global top dishes returned: {len(discover_recommendations)}")
for idx, rec in enumerate(discover_recommendations, start=1):
    name = rec.get("name", "<unknown>")
    restaurant_name = rec.get("restaurant_name", "<unknown>")
    score = rec.get("score")
    print(f"{idx}. {name} @ {restaurant_name} (score={score})")

NameError: name 'nearby_payload' is not defined

## Internal Cache Note
Cache stats and manual invalidation are internal operational controls and are intentionally not exposed as public API endpoints.